In [1]:
### Load the environment variables

from dotenv import load_dotenv

load_dotenv()

True

#### ***1.Document Ingestion***

#### ***Load the all files and convert that data into the LangChain Document Object.***

In [2]:
###path Exists
import os
path = "../kubernetes"

if os.path.exists(path):
    print("Path Is Exist.")
else:
    print("Path is not Exist.")

Path Is Exist.


In [3]:
### Load all pdf files using DirectoryLoader by PyMuPdfLoader

from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()


print("Number Of Documents:",len(documents))

C:\Users\mukko\AppData\Local\Temp\ipykernel_7828\2542035176.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader


Number Of Documents: 3983


#### ***2.Document Splitting***

##### **Document Splitting is a process of split documents into chunks based on chunk_size and chunk overlap using Recursive Character Text Splitter, it will preseve the structure and meaning of the data like paragraphs, lines,sentences, and more.**

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 12694


In [5]:
### Lets test the single chunk randomly

chunks[9]

Document(metadata={'producer': 'WeasyPrint 56.1', 'creator': '', 'creationdate': '', 'source': '..\\kubernetes\\Concepts.pdf', 'file_path': '..\\kubernetes\\Concepts.pdf', 'total_pages': 676, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 2}, page_content='properties to share the Operating System (OS) among the applications. Therefore, containers\nare considered lightweight. Similar to a VM, a container has its own filesystem, share of CPU,\nmemory, process space, and more. As they are decoupled from the underlying infrastructure,\nthey are portable across clouds and OS distributions.\nContainers have become popular because they provide extra benefits, such as:\nAgile application creation and deployment: increased ease and efficiency of container\nimage creation compared to VM image use.\nContinuous development, integration, and deployment: provides for reliable and frequent')

In [6]:
### page content
chunks[9].page_content

'properties to share the Operating System (OS) among the applications. Therefore, containers\nare considered lightweight. Similar to a VM, a container has its own filesystem, share of CPU,\nmemory, process space, and more. As they are decoupled from the underlying infrastructure,\nthey are portable across clouds and OS distributions.\nContainers have become popular because they provide extra benefits, such as:\nAgile application creation and deployment: increased ease and efficiency of container\nimage creation compared to VM image use.\nContinuous development, integration, and deployment: provides for reliable and frequent'

In [7]:
### metadata
chunks[9].metadata

{'producer': 'WeasyPrint 56.1',
 'creator': '',
 'creationdate': '',
 'source': '..\\kubernetes\\Concepts.pdf',
 'file_path': '..\\kubernetes\\Concepts.pdf',
 'total_pages': 676,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '',
 'trapped': '',
 'modDate': '',
 'creationDate': '',
 'page': 2}

#### ***3.Embeddings.***
##### **Embedding is a numerical representation of actual text.**

In [8]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [9]:
### Using batching technique to create embedding vector for chunks
from langchain_chroma import Chroma
batch_size = 200
batch_chunk = chunks[:200]
vectorstore = Chroma.from_documents(
    documents = batch_chunk,
    embedding=embedding_model,
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag"
)

In [10]:

for i in range(batch_size,len(chunks),batch_size):

    batch_chunk = chunks[i:i+batch_size]

    vectorstore.add_documents(batch_chunk)

    min_dx = min(i+batch_size,len(chunks))

    print(f"Added vectors succesfully from {i} to {min_dx-1}")
    
print("Added all vectors succesfully")

Added vectors succesfully from 200 to 399
Added vectors succesfully from 400 to 599
Added vectors succesfully from 600 to 799
Added vectors succesfully from 800 to 999
Added vectors succesfully from 1000 to 1199
Added vectors succesfully from 1200 to 1399
Added vectors succesfully from 1400 to 1599
Added vectors succesfully from 1600 to 1799
Added vectors succesfully from 1800 to 1999
Added vectors succesfully from 2000 to 2199
Added vectors succesfully from 2200 to 2399
Added vectors succesfully from 2400 to 2599
Added vectors succesfully from 2600 to 2799
Added vectors succesfully from 2800 to 2999
Added vectors succesfully from 3000 to 3199
Added vectors succesfully from 3200 to 3399
Added vectors succesfully from 3400 to 3599
Added vectors succesfully from 3600 to 3799
Added vectors succesfully from 3800 to 3999
Added vectors succesfully from 4000 to 4199
Added vectors succesfully from 4200 to 4399
Added vectors succesfully from 4400 to 4599
Added vectors succesfully from 4600 to 4

In [11]:
vectorstore._collection.count()

12694

In [12]:
question = "What is a Kubernetes Deployment?"

In [14]:
vector = embedding_model.embed_query(question)
print(len(vector))

1024


In [13]:
docs = vectorstore.similarity_search(question,k=5)
docs

[Document(id='525d38e4-acbe-4a4d-8f52-9d30706d0de8', metadata={'producer': 'WeasyPrint 56.1', 'creationDate': '', 'subject': '', 'source': '..\\kubernetes\\Tutorials.pdf', 'creationdate': '', 'page': 7, 'format': 'PDF 1.7', 'modDate': '', 'author': '', 'moddate': '', 'title': '', 'keywords': '', 'trapped': '', 'total_pages': 168, 'file_path': '..\\kubernetes\\Tutorials.pdf', 'creator': ''}, page_content='directly onto specific machines as packages deeply integrated into the host. Kubernetes\nautomates the distribution and scheduling of application containers across a cluster in\na more efficient way. Kubernetes is an open-source platform and is production-ready.\nA Kubernetes cluster consists of two types of resources:\nThe Control Plane coordinates the cluster\nNodes are the workers that run applications\nSummary:\nKubernetes cluster\n• \n• \n• \n• \n• \n•'),
 Document(id='f3ed5040-a141-47cc-944b-ae87381abb19', metadata={'file_path': '..\\kubernetes\\Tutorials.pdf', 'total_pages': 168